# Day 4: Architecture and MongoDB
**Frontend, backend and APIs, Matplotlib, NoSQL and BSON, MongoDB CRUD, indexes and data modelling**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup
Colab uses **mongomock**, an in-memory practice MongoDB. VS Code can use a real MongoDB (local or a free Atlas cluster). The CRUD code is identical.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q mongomock pymongo
import mongomock
client = mongomock.MongoClient()

# ===== Option B: VS Code with real MongoDB (uncomment to use) =====
# %pip install pymongo
# import os
# from pymongo import MongoClient
# uri = os.environ.get("MONGO_URI", "mongodb://localhost:27017")   # or your Atlas URI, from an env variable
# client = MongoClient(uri)

db = client["shop"]
products = db["products"]
products.delete_many({})       # start fresh each run
print("Connected. Collections:", db.list_collection_names())

# Session 1: Architecture and charts
## 2. A request's journey, in code
A tiny simulation of client, middleware, route, service and repository. Each function has one job.

In [ ]:
FAKE_DB = {"tickets": [{"id": 1, "customer": "Mei", "status": "open"}]}

def repository_list(status):                 # data access only
    return [t for t in FAKE_DB["tickets"] if t["status"] == status]

def service_open_tickets():                  # business rules
    return sorted(repository_list("open"), key=lambda t: t["id"])

def route_get_tickets(request):              # receives the request, returns a response
    return {"status": 200, "body": service_open_tickets()}

def middleware(handler):                     # runs on every request
    def wrapped(request):
        print("LOG", request["method"], request["path"])
        if request.get("token") != "demo":
            return {"status": 401, "body": "unauthorised"}
        return handler(request)
    return wrapped

api = middleware(route_get_tickets)
print(api({"method": "GET", "path": "/tickets", "token": "demo"}))
print(api({"method": "GET", "path": "/tickets"}))

## 3. Matplotlib basics

In [ ]:
import matplotlib.pyplot as plt

months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
tickets_2025 = [120, 150, 90, 180, 210, 190]
tickets_2026 = [140, 160, 130, 200, 260, 240]
plt.figure(figsize=(7, 4))
plt.plot(months, tickets_2025, marker="o", label="2025")
plt.plot(months, tickets_2026, marker="o", label="2026")
plt.title("Support tickets per month")
plt.xlabel("Month")
plt.ylabel("Tickets")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cats = ["Food", "Rent", "Travel", "Fun"]
amounts = [6000, 15000, 2500, 3000]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
bars = ax1.bar(cats, amounts, color="#0FA3A3")
ax1.bar_label(bars)
ax1.set_title("Expenses by category")
ax2.pie(amounts, labels=cats, autopct="%1.0f%%")
ax2.set_title("Share of spending")
fig.tight_layout()
plt.savefig("expenses.png")
plt.show()

### TODO 1: Plot a histogram of these resolution times (hours) with 5 bins: `[4, 7, 2, 8, 9, 5, 3, 6, 8, 10, 1]`

In [ ]:
# TODO: write your code here

# Session 2: NoSQL and MongoDB
## 4. Documents, BSON and ObjectId

In [ ]:
from bson import ObjectId
from datetime import datetime, timezone

oid = ObjectId()
print("ObjectId:", oid, "| created at:", oid.generation_time)

doc = {"name": "Mechanical Keyboard", "price": 3499, "tags": ["electronics", "office"],
       "stock": {"bengaluru": 12, "tokyo": 4}, "added": datetime.now(timezone.utc)}
result = products.insert_one(doc)
print("Inserted _id:", result.inserted_id, type(result.inserted_id).__name__)
print(products.find_one({"_id": result.inserted_id}, {"_id": 0}))

**MongoDB Compass (optional):** install Compass, paste your Atlas or local connection string, then browse the `shop` database. In Colab, mongomock lives only in memory, so Compass cannot see it.

# Session 3: CRUD, indexes and data modelling
## 5. Create: insert documents

In [ ]:
products.delete_many({})
result = products.insert_one({"name": "Pen", "category": "Stationery", "price": 20, "stock": 100})
print("New id:", result.inserted_id)

products.insert_many([
    {"name": "Notebook", "category": "Stationery", "price": 80, "stock": 40},
    {"name": "Mouse", "category": "Electronics", "price": 500, "stock": 15},
    {"name": "Keyboard", "category": "Electronics", "price": 1500, "stock": 4},
    {"name": "Monitor", "category": "Electronics", "price": 9000, "stock": 3},
    {"name": "Mug", "category": "Home", "price": 250, "stock": 0, "tags": ["gift", "kitchen"]},
    {"name": "Lamp", "category": "Home", "price": 1200, "stock": 7},
])
print("Total documents:", products.count_documents({}))

## 6. Read: find with filters and projections

In [ ]:
print(products.find_one({"name": "Mouse"}))

print("\nUnder 1000:")
for p in products.find({"price": {"$lt": 1000}}, {"_id": 0, "name": 1, "price": 1}):
    print(" ", p)

print("\nElectronics or Home with stock < 5:")
query = {"category": {"$in": ["Electronics", "Home"]}, "stock": {"$lt": 5}}
for p in products.find(query):
    print(" ", p["name"], p["stock"])

print("\nTop 3 most expensive:")
for p in products.find().sort("price", -1).limit(3):
    print(" ", p["name"], p["price"])

## 7. Update and delete

In [ ]:
products.update_one({"name": "Pen"}, {"$set": {"price": 25}})
for p in products.find({"category": "Electronics"}):      # +10% price (mongomock has no $mul)
    products.update_one({"_id": p["_id"]}, {"$set": {"price": round(p["price"] * 1.1)}})
products.update_many({"stock": 0}, {"$set": {"active": False}})
products.update_one({"name": "Mug"}, {"$push": {"tags": "ceramic"}})

print("Will delete:", [p["name"] for p in products.find({"active": False})])   # check first!
res = products.delete_many({"active": False})
print("Deleted:", res.deleted_count, "| Remaining:", products.count_documents({}))

## 8. Indexes
Index the fields you filter and sort on. A unique index stops duplicates.

In [ ]:
from pymongo.errors import DuplicateKeyError

products.create_index("category")
products.create_index([("category", 1), ("price", -1)])
products.create_index("sku", unique=True, sparse=True)
print("Indexes:", list(products.index_information().keys()))

products.update_one({"name": "Pen"}, {"$set": {"sku": "P-100"}})
try:
    products.insert_one({"name": "Pen copy", "category": "Stationery", "price": 20, "stock": 1, "sku": "P-100"})
except DuplicateKeyError:
    print("Blocked duplicate sku P-100")

## 9. Data modelling: embed vs reference
A ticket **embeds** its notes (small, read together) and **references** its customer by `_id` (shared by many tickets).

In [ ]:
customers = db["customers"]
customers.delete_many({})
mei_id = customers.insert_one({"name": "Mei", "plan": "5G Unlimited", "city": "Osaka"}).inserted_id

ticket = {"customer_id": mei_id,                         # reference
          "issue": "No signal at home", "priority": "high", "status": "open",
          "notes": [{"by": "Kenji", "text": "Checked tower status"},      # embedded
                    {"by": "Kenji", "text": "Asked customer to restart"}]}
tid = db["support"].insert_one(ticket).inserted_id

t = db["support"].find_one({"_id": tid})
c = customers.find_one({"_id": t["customer_id"]})        # second lookup for the reference
print(f'{c["name"]} ({c["plan"]}): {t["issue"]}, {len(t["notes"])} notes')

## 10. From database to chart

In [ ]:
value_by_cat = {}
for p in products.find():
    value_by_cat[p["category"]] = value_by_cat.get(p["category"], 0) + p["price"] * p["stock"]
print(value_by_cat)

plt.figure(figsize=(6, 4))
plt.bar(value_by_cat.keys(), value_by_cat.values(), color=["#0FA3A3", "#FF8A3D", "#0B5D63"])
plt.title("Stock value by category")
plt.ylabel("Value")
plt.show()

## 11. SmartDesk step 3: tickets in MongoDB, chart by priority

In [ ]:
import random
from datetime import timedelta

tickets = db["tickets"]
tickets.delete_many({})
tickets.create_index("status")
tickets.create_index("ticket_no", unique=True)

random.seed(4)
names = ["Mei", "Ravi", "Aiko", "Kenji", "Priya", "Hana", "Arjun", "Yuki"]
issues = ["No signal at home", "Charged twice", "Cannot log in", "Slow data", "SIM not activated"]
now = datetime.now(timezone.utc)
tickets.insert_many([{
    "ticket_no": f"T-{i:03d}", "customer": random.choice(names), "issue": random.choice(issues),
    "priority": random.choice(["low", "medium", "medium", "high"]),
    "status": random.choice(["open", "open", "closed"]),
    "created": now - timedelta(days=random.randint(0, 14))} for i in range(1, 21)])
print("Tickets:", tickets.count_documents({}), "| open:", tickets.count_documents({"status": "open"}))

counts = {}
for t in tickets.find({"status": "open"}):
    counts[t["priority"]] = counts.get(t["priority"], 0) + 1
order = [p for p in ["high", "medium", "low"] if p in counts]
print({p: counts[p] for p in order})

plt.figure(figsize=(5, 3.5))
bars = plt.bar(order, [counts[p] for p in order], color=["#FF8A3D", "#0FA3A3", "#0B5D63"][:len(order)])
plt.bar_label(bars)
plt.title("Open SmartDesk tickets by priority")
plt.savefig("tickets_by_priority.png")
plt.show()

## 12. A layered design: repository class

In [ ]:
class TicketRepository:
    def __init__(self, collection):
        self.col = collection

    def add(self, ticket):
        return self.col.insert_one(ticket).inserted_id

    def list_by_status(self, status):
        return list(self.col.find({"status": status}, {"_id": 0, "ticket_no": 1, "customer": 1}))

    def close(self, ticket_no):
        return self.col.update_one({"ticket_no": ticket_no}, {"$set": {"status": "closed"}}).modified_count

repo = TicketRepository(tickets)
repo.add({"ticket_no": "T-021", "customer": "Mei", "issue": "Roaming not working", "priority": "high",
          "status": "open", "created": now})
print("Open:", len(repo.list_by_status("open")), "| closed T-021:", repo.close("T-021"))

### TODO 2: Add a `delete_by_ticket_no` method to the repository and test it.

In [ ]:
# TODO: write your code here

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)